# 2. Scales, the grid-free domain and extended persistence

**Scales.** The size of collective structures varies, so every field is computed at $M=4$ widths
$\sigma_1<\dots<\sigma_4$ derived from the data: $\sigma_{\min}=\max(d_{\min}/2, 1\,\mathrm{cm})$, with $d_{\min}$ the
smallest distance between two robots anywhere in the collection, and $\sigma_{\max}=D_s/(2.35\,k)$, with $D_s$ the
largest diameter of the swarm (two footprints stop being resolved at $d\le 2.35\sigma$). The cap $k$ is chosen without
labels, as the first cap at which the plateau of the resolution map levels off ($k=10$; Appendix C).

In [ ]:
import sys, pickle
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT / "analysis"), str(ROOT / "src"), str(ROOT)]
import numpy as np
import matplotlib.pyplot as plt
import figlib as L
L.style()
PAPER = ROOT / "results" / "epuck" / "paper"
RES = ROOT / "results" / "epuck"
load = lambda p: pickle.load(open(p, "rb"))

In [ ]:
import make_figures as M
ladder = load(RES / "scale_ladder.pkl")["ladder"]
print("scale ladder (cm):", np.round(100 * np.asarray(ladder), 2))
M.fig_scales(); M.fig_uncapped()
from IPython.display import Image, display
display(Image(filename=str(ROOT / "figures/fig_scales.png"), width=650))
display(Image(filename=str(ROOT / "figures/figA_uncapped.png"), width=900))

The second figure shows why the ladder is capped: on an uncapped ladder that reaches $D_s/2.35$ every field becomes a
single featureless bump.

## The grid-free domain

A pixel grid would impose a chosen resolution. Each recording gets its own domain at
each scale, built from three families of seeds:

1. a **subset of the robot positions** over the recording, chosen by farthest-point sampling until no robot position lies farther than $\sigma/2$ from it (finer at small scales);
2. **near-critical seeds**: barycentres of the 5 % Delaunay triangles where each field's gradient is smallest;
3. **coupling skeleton**: up to 150 points where $J_\sigma=0$, found by 12 bisection steps on Delaunay edges.

The five fields are evaluated exactly at every seed, and seeds farther than $5\sigma$ from every robot are discarded;
the domain is the alpha complex of the remaining seeds, truncated at squared radius $(3\sigma)^2$.

In [ ]:
import gudhi
from matplotlib.collections import LineCollection
d = load(PAPER / "top1_tracks.pkl"); idx = d["idx"]
tax = load(PAPER / "taxonomy.pkl")[1] if (PAPER / "taxonomy.pkl").exists() else None
e = int(tax["exemplars"][0]) if tax is not None else 175
si = 2; sig = float(ladder[si])
st1 = load(RES / "stage1_alpha_fields" / f"traj{int(idx[e]):04d}_s{si}.pkl")
seeds = st1["seeds"]
core, bary, skel = M._seed_families(seeds, np.asarray(st1["fields"]["J"]), d["tracks"][e]["pos"])
print(f"{len(seeds)} seeds: {core.sum()} robot positions, {bary.sum()} near-critical, {skel.sum()} on J = 0")
st = gudhi.AlphaComplex(points=seeds).create_simplex_tree(max_alpha_square=(3 * sig) ** 2)
segs = [seeds[s] for s, _ in st.get_skeleton(1) if len(s) == 2]
fig, ax = plt.subplots(figsize=(5, 5))
ax.add_collection(LineCollection(segs, colors="#c3c2b7", linewidths=0.3))
ax.scatter(*seeds[core].T, s=5, c=L.INK2, label="robot positions")
ax.scatter(*seeds[bary].T, s=12, c=L.MCOL[1], label="near-critical")
ax.scatter(*seeds[skel].T, s=10, c=L.MCOL[0], label="$J=0$ skeleton")
ax.set_aspect("equal"); ax.legend(loc="upper right"); plt.show()

## Why a fifth field, $J_\sigma$

Persistence describes **one scalar field at a time**. Taken separately, the diagrams of $\varphi_\sigma$ and $\psi_\sigma$ record how much
expansion and how much rotation there is, but not *where they are relative to each other*: two recordings can have the
same diagrams for each potential and differ in the alignment of sources and vortices. $J_\sigma=\sigma^2
\det(\nabla\varphi_\sigma,\nabla\psi_\sigma)$ measures that alignment: zero where the gradients are parallel, extreme where
they cross at right angles. It records in one field what the separate diagrams of the two potentials miss; a joint summary of pairs of fields
would not need it.

## Extended persistence

On the fixed complex each field defines a **lower-star filtration** (a simplex enters at the largest field value of its
vertices). Ordinary persistence leaves the main features with infinite lifetimes; **extended persistence** continues with
a descending sweep, so every feature is finite. Each field gives six diagrams: Ord$_0$, Ord$_1$ (ascending sweep),
Rel$_1$, Rel$_2$ (relative sweep), Ext$_0$, Ext$_1$ (born in the first sweep, killed in the second). Ext$_0$ pairs the global minimum with the global
maximum, and Ext$_1$ records the holes of the domain.

In [ ]:
st2 = load(RES / "stage2_alpha_dgms" / f"traj{int(idx[e]):04d}_s{si}.pkl")["dgms"]
names = ["Ord0", "Ord1", "Rel1", "Rel2", "Ext0", "Ext1"]
for f in L.FIELDS:
    print(f"{f:4s}", "  ".join(f"{n}:{len(x):3d}" for n, x in zip(names, st2[f])))
fig, axes = plt.subplots(1, 5, figsize=(14, 3))
for ax, f in zip(axes, L.FIELDS):
    allv = np.concatenate([np.asarray(x).ravel() for x in st2[f] if len(x)])
    lo, hi = allv.min(), allv.max()
    ax.plot([lo, hi], [lo, hi], color=L.GRID)
    for k, x in enumerate(st2[f]):
        x = np.asarray(x)
        if len(x): ax.scatter(x[:, 0], x[:, 1], s=12, marker=M.DGM_MARK[k], label=names[k])
    ax.set_title(f); ax.set_xlabel("birth")
axes[0].set_ylabel("death"); axes[-1].legend(fontsize=7); plt.show()

### Recomputing the diagrams from the seeds

The next cell recomputes the extended persistence of $\rho_\sigma$ from the seeds and the field values with GUDHI, and
checks that it matches the stored diagrams.

In [ ]:
from gaussian_splatting.alpha import AlphaSimplexTree
from gaussian_splatting.persistence import PersistenceComputer
pc = PersistenceComputer(AlphaSimplexTree(seeds, alpha_sq_max=(3 * sig) ** 2))
dg = pc.compute_field(np.asarray(st1["fields"]["rho"]))
print("identical to the stored diagrams:", all(np.allclose(np.sort(a, 0), np.sort(np.asarray(b), 0)) for a, b in zip(dg, st2["rho"])))